# Deterministic 3D Pipe Autorouting

**Tuba v4 Course — Notebook 05 of 08**

---

Routing a pipe between two endpoints while dodging equipment and keepout zones
is one of the most tedious tasks in piping design. Tuba's **AutoroutingAgent**
automates this with A\*-based pathfinding on a 3D occupancy grid.

### What you'll learn

| Step | Topic |
|------|-------|
| 1 | Define obstacles (keepout volumes) inside a `Model` |
| 2 | Configure a `PipeRouteRequest` with start / goal endpoints and constraints |
| 3 | Run `AutoroutingAgent` to generate multiple candidate routes |
| 4 | Visualize the candidates interactively with PyVista |
| 5 | Export Code_Aster study files for downstream stress analysis |

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tuba import Model
from tuba.routing import AutoroutingAgent, GridRouter
from tuba.routing.solver_loop import SolverLoopConfig
from tuba.routing.types import (
    PipeRouteRequest,
    RouteEndpoint,
    RoutingConstraints,
    RoutingGridSpec,
)
from tuba.routing.visualization import build_route_plotter, export_route_scene_html

import pyvista as pv
# Defaults to zoomable 'client' locally; set TUBA_NOTEBOOK_BACKEND=static for nbconvert/CI.
from tuba.visualizer.notebook import configure_notebook_backend
JUPYTER_BACKEND = configure_notebook_backend()

## 1 — Define the Scene

The autorouter operates on a standard `Model`. We add the usual material and
pipe section, then place **obstacle keepout volumes** — axis-aligned cuboids
that the router must steer around.

Think of obstacles as bounding boxes around equipment, cable trays, or
maintenance-access zones that the pipe cannot penetrate.

In [ ]:
# ── Model, material & section ────────────────────────────────────────
model = Model("AutoroutingDemo")

model.add_material(
    "steel",
    E=210e9,
    nu=0.3,
    rho=7850,
    alpha=12e-6,
    allowable_stress={20.0: 140e6, 120.0: 125e6},
)

model.add_pipe_section("DN100", OD=0.1143, WT=0.00602)

# ── Load case ─────────────────────────────────────────────────────────
model.define_load_case("Hot", gravity=True, pressure=1e6, temperature=120.0)

# ── Obstacles ─────────────────────────────────────────────────────────
model.add_obstacle(
    id="equipment_box",
    type="cuboid",
    min_point=[1.5, -0.4, -0.4],
    max_point=[2.5,  0.4,  0.4],
)

model.add_obstacle(
    id="maintenance_keepout",
    type="cuboid",
    min_point=[2.8, 0.8, -0.4],
    max_point=[3.4, 1.4,  0.8],
)

print(f"Model '{model.project_name}' ready — {len(model.obstacles)} obstacles defined.")

## 2 — Define the Route Request

A `PipeRouteRequest` tells the router *what* to connect:

| Field | Purpose |
|-------|---------|
| `start` / `goal` | `RouteEndpoint` — named 3D coordinates |
| `section` | Pipe section name (must exist in the model) |
| `material` | Material name (must exist in the model) |
| `constraints` | `RoutingConstraints` — clearance envelope around obstacles, minimum bend radius |

In [ ]:
request = PipeRouteRequest(
    id="P-100",
    start=RouteEndpoint("A", (0.0, 0.0, 0.0)),
    goal=RouteEndpoint("B", (4.0, 0.0, 0.0)),
    section="DN100",
    material="steel",
    constraints=RoutingConstraints(
        clearance=0.10,
        min_bend_radius=0.20,
    ),
)

print(f"Route request '{request.id}':")
print(f"  Start  {request.start.id} → {request.start.point}")
print(f"  Goal   {request.goal.id} → {request.goal.point}")
print(f"  Clearance      = {request.constraints.clearance} m")
print(f"  Min bend radius = {request.constraints.min_bend_radius} m")

## 3 — Visualize the Scene Before Routing

`build_route_plotter` renders the model's obstacles, the start / goal
endpoints, and any existing pipe geometry in a single PyVista scene.
This lets you sanity-check the layout before committing to a solve.

In [ ]:
plotter = build_route_plotter(model, request=request)
plotter.show(jupyter_backend="client")

## 4 — Run the Autorouter

The **AutoroutingAgent** orchestrates two components:

| Component | Role |
|-----------|------|
| `GridRouter` | Builds a 3D occupancy grid, inflates obstacles by the clearance envelope, and runs A\* pathfinding to produce `candidate_count` routes. |
| `SolverLoopConfig` | Controls downstream FEA. Setting `run_solver=False` and `export_study=True` writes Code_Aster `.comm`, `.mail`, and `.export` files without launching the solver — perfect for batch submission on a cluster. |

`RoutingGridSpec` controls the grid:
- **`cell_size`** — edge length of each voxel (smaller → finer paths, slower).
- **`margin`** — padding around the bounding box of endpoints + obstacles.

In [ ]:
OUTPUT_ROOT = Path(".").resolve().parent / "routing_reports" / "notebook_demo"

agent = AutoroutingAgent(
    router=GridRouter(
        RoutingGridSpec(cell_size=0.25, margin=1.0),
        candidate_count=3,
    ),
    solver_config=SolverLoopConfig(
        run_solver=False,
        export_study=True,
    ),
    output_root=str(OUTPUT_ROOT),
)

run = agent.route_pipe(model, request, apply=True)

print(f"Routing complete.")
print(f"  Candidates generated : {len(run.result.candidates)}")
print(f"  Selected index       : {run.result.selected_index}")
print(f"  Output directory     : {OUTPUT_ROOT}")

## 5 — Visualize Route Candidates

Passing the `result` object back into `build_route_plotter` overlays all
candidate paths on the scene:

- **Selected route** — solid green/teal line.
- **Alternatives** — ghosted (semi-transparent) lines.

Rotate the scene to inspect how each candidate navigates around the obstacles.

In [ ]:
plotter = build_route_plotter(model, request=request, result=run.result)
plotter.show(jupyter_backend="client")

## 6 — Export Interactive HTML

Share the 3D route scene with colleagues who don't have Python installed.
`export_route_scene_html` writes a self-contained HTML file with embedded
WebGL rendering.

In [ ]:
html_path = export_route_scene_html(
    model,
    "autoroute_scene.html",
    request=request,
    result=run.result,
)
print(f"Interactive route scene exported to: {html_path}")

## 7 — Review Generated Study Files

Because we set `export_study=True`, the agent wrote Code_Aster input files
for each candidate route:

| File | Purpose |
|------|---------|
| `.comm` | Code_Aster command file (solver instructions) |
| `.mail` | Mesh file (nodes, elements, groups) |
| `.export` | Job descriptor (ties `.comm` + `.mail` together) |

These can be submitted directly to a Code_Aster installation or HPC cluster.

In [ ]:
# ── List generated study files ────────────────────────────────────────
if OUTPUT_ROOT.exists():
    print(f"Study files in: {OUTPUT_ROOT}\n")
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file():
            rel = path.relative_to(OUTPUT_ROOT)
            size_kb = path.stat().st_size / 1024
            print(f"  {rel}  ({size_kb:.1f} KB)")
else:
    print("No study files found — the solver config may have skipped export.")

---

## Key Takeaways

| Concept | Detail |
|---------|--------|
| **A\* pathfinding** | Runs on a 3D occupancy grid built from the model's obstacle list |
| **Obstacle avoidance** | Each obstacle is inflated by a `clearance` envelope before grid marking |
| **Multiple candidates** | `candidate_count` controls how many alternative routes are generated, ranked by total length and bend count |
| **Code_Aster export** | `.comm` / `.mail` / `.export` files are written per candidate for downstream stress analysis |

**Next →** [Notebook 06](06_visualization.ipynb) dives into the visualization and reporting pipeline.